<a href="https://colab.research.google.com/github/psept77-star/CAPA_Effectiveness_Prediction/blob/main/CAPA_Effectiveness_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
# Import liabraries
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


In [17]:
# STEP 1: HISTORICAL DATA LOADING & ENCODING

# Load the data directly using the correct excel reader
df = pd.read_excel('/content/sample_data/CAPA_Dummy_Dataset_4000.xlsx')
display(df.head())

# Initialize unique LabelEncoders to convert raw strings into integers natively
dept_encoder = LabelEncoder()
sev_encoder = LabelEncoder()
rc_encoder = LabelEncoder()
overdue_encoder = LabelEncoder()
recurrence_encoder = LabelEncoder()

df["Dept_encoded"] = dept_encoder.fit_transform(df["Department"])
df["Sev_encoded"] = sev_encoder.fit_transform(df["Severity"])
df["RC_encoded"] = rc_encoder.fit_transform(df["Root_Cause"])
df["Overdue_encoded"] = overdue_encoder.fit_transform(df["Overdue"])
df["Recurrence_encoded"] = recurrence_encoder.fit_transform(df["Recurrence_90d"])

,CAPA_ID,Department,Severity,Root_Cause,Closure_Time_Days,Overdue,Recurrence_90d,CAPA_Effective
0,CAPA-0001,Quality Assurance,Major,Equipment Failure,93,Yes,Yes,0
1,CAPA-0002,Logistics,Major,Equipment Failure,48,Yes,No,1
2,CAPA-0003,Packaging,Major,Human Error,76,Yes,No,1
3,CAPA-0004,Quality Assurance,Minor,Software Bug,6,No,No,1
4,CAPA-0005,Production,Minor,SOP Lack of Clarity,2,No,Yes,0


### Model Comparison & Performance Evaluation
1. **Logistic Regression**
2. **Support Vector Machine (SVM)**
3. **XGBoost Classifier**
4. **Random Forest Classifier**

We will evaluate each model using Multi-Output classification on the test dataset.

In [18]:
# Configure models in a simple dictionary
clf_models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "SVM": SVC(random_state=42),
    "XGBoost": xgb.XGBClassifier(random_state=42, eval_metric='logloss'),
    "Random Forest": RandomForestClassifier(random_state=42)
}

comparison_results = []

for name, model in clf_models.items():
    # Wrap with MultiOutput to handle multiple targets seamlessly
    multi_model = MultiOutputClassifier(model).fit(X_train_c, y_train_c)
    preds = multi_model.predict(X_test_c)

    comparison_results.append({
        "Model": name,
        "Acc (Effective)": accuracy_score(y_test_c["CAPA_Effective"], preds[:, 0]),
        "Acc (Recurrence)": accuracy_score(y_test_c["Recurrence_encoded"], preds[:, 1]),
        "F1 (Effective)": f1_score(y_test_c["CAPA_Effective"], preds[:, 0], average='macro'),
        "F1 (Recurrence)": f1_score(y_test_c["Recurrence_encoded"], preds[:, 1], average='macro')
    })

# Display the structured results table
display(pd.DataFrame(comparison_results).set_index("Model"))

,Acc (Effective),Acc (Recurrence),F1 (Effective),F1 (Recurrence)
Model,,,,
Logistic Regression,0.63125,0.75625,0.512372,0.430605
SVM,0.63625,0.75625,0.500548,0.430605
XGBoost,0.62375,0.74875,0.540211,0.446889
Random Forest,0.62875,0.74500,0.540247,0.449771


In [19]:
# STEP 2: MULTI-TARGET ARCHITECTURE & TRAINING

print("--- Training Multi-Objective Quality Intelligence Model ---")

# Define the features known at the time of drafting
classification_features = ["Dept_encoded", "Sev_encoded", "RC_encoded", "Overdue_encoded"]

X_clf = df[classification_features]
# We are training the model to map patterns to TWO targets at once:
y_clf = df[["CAPA_Effective", "Recurrence_encoded"]]

# Train/Test Split for the Classifier
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42)

# Build a Multi-Output Random Forest Classifier
clf_model = RandomForestClassifier(n_estimators=100, random_state=42)
clf_model.fit(X_train_c, y_train_c)

# --- Regression Model to Predict the Cycle Time (Closure Days) ---
# For cycle time, we can include the actual target labels as features since the regressor
# estimates implementation speed based on the quality profile of the plan
regression_features = ["Dept_encoded", "Sev_encoded", "RC_encoded", "Overdue_encoded", "CAPA_Effective"]
X_reg = df[regression_features]
y_reg = df["Closure_Time_Days"]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

reg_model = RandomForestRegressor(n_estimators=100, random_state=42)
reg_model.fit(X_train_r, y_train_r)

print("✅ Systems successfully mapped to historical data for CAPA.\n")


--- Training Multi-Objective Quality Intelligence Model ---
✅ Systems successfully mapped to historical data for CAPA.



In [20]:
# STEP 3: ADVANCED LIVE INFERENCE PIPELINE

def analyze_upcoming_capa_risk(department, severity, root_cause, is_overdue):
    """Predicts effectiveness, 90-day recurrence risk, and estimated completion time."""
    try:
        # Encode inputs on the fly
        encoded_dept = dept_encoder.transform([department])[0]
        encoded_sev = sev_encoder.transform([severity])[0]
        encoded_rc = rc_encoder.transform([root_cause])[0]
        encoded_overdue = overdue_encoder.transform([is_overdue])[0]

        # 1. Predict Effectiveness and Recurrence
        clf_input = pd.DataFrame([{
            "Dept_encoded": encoded_dept,
            "Sev_encoded": encoded_sev,
            "RC_encoded": encoded_rc,
            "Overdue_encoded": encoded_overdue
        }])

        clf_preds = clf_model.predict(clf_input)[0]
        clf_probs = clf_model.predict_proba(clf_input)

        pred_effective = clf_preds[0]
        pred_recurrence = clf_preds[1]

        # Calculate individual confidence levels
        eff_confidence = clf_probs[0][0][pred_effective] * 100
        rec_confidence = clf_probs[1][0][pred_recurrence] * 100

        # 2. Predict Actual Closure Time based on estimated quality profile
        reg_input = pd.DataFrame([{
            "Dept_encoded": encoded_dept,
            "Sev_encoded": encoded_sev,
            "RC_encoded": encoded_rc,
            "Overdue_encoded": encoded_overdue,
            "CAPA_Effective": pred_effective
        }])

        predicted_days = reg_model.predict(reg_input)[0]

        # Translate values back to human labels
        eff_label = " EFFECTIVE" if pred_effective == 1 else "🔴 INEFFECTIVE (Rework Risk)"
        rec_label = " HIGH RISK" if recurrence_encoder.classes_[pred_recurrence] == "Yes" else "✅ LOW RISK"

        print("==================================================")
        print("      ADVANCED QUALITY INTELLIGENCE ENGINE     ")
        print("==================================================")
        print(f" DRAFT CAPA: {severity} profile for {department}")
        print(f" Causal Origin: {root_cause} (Overdue Status: {is_overdue})")
        print("--------------------------------------------------")
        print(f" 1. ACTION PLAN VIABILITY:")
        print(f"   -> {eff_label} ({eff_confidence:.1f}% Model Confidence)")
        print(f" 2. 90-DAY DEVIATION RECURRENCE:")
        print(f"   -> {rec_label} ({rec_confidence:.1f}% Model Confidence)")
        print(f" 3. FORECASTED CYCLE TIME:")
        print(f"   -> Estimated {int(predicted_days)} days to verify and close")
        print("==================================================\n")

    except ValueError as e:
        print(f"❌ Input categorization error: {e}")


In [21]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Create interactive widgets
dept_dropdown = widgets.Dropdown(
    options=list(dept_encoder.classes_),
    description='Department:',
    style={'description_width': 'initial'}
)

sev_dropdown = widgets.Dropdown(
    options=list(sev_encoder.classes_),
    description='Severity:',
    style={'description_width': 'initial'}
)

rc_text = widgets.Text(
    value='',
    placeholder='e.g., Human Error, Equipment Failure',
    description='Root Cause:',
    style={'description_width': 'initial'}
)

overdue_dropdown = widgets.Dropdown(
    options=['No', 'Yes'],
    description='CAPA Overdue?:',
    style={'description_width': 'initial'}
)

predict_button = widgets.Button(
    description='🔮 Predict',
    button_style='primary',
    tooltip='Click to run risk assessment'
)

output_area = widgets.Output()

def on_predict_clicked(b):
    with output_area:
        clear_output()
        # Clean user input for root cause to match expected format
        rc_val = rc_text.value.strip()

        # Check if the typed root cause exists in our encoder
        if rc_val not in rc_encoder.classes_:
            print(f"❌ Error: '{rc_val}' is not a valid Root Cause.")
            print(f"Valid options are: {list(rc_encoder.classes_)}")
            return

        # Call the live pipeline function
        analyze_upcoming_capa_risk(
            department=dept_dropdown.value,
            severity=sev_dropdown.value,
            root_cause=rc_val,
            is_overdue=overdue_dropdown.value
        )

predict_button.on_click(on_predict_clicked)

# Arrange and display the widget layout
input_form = widgets.VBox([
    widgets.HTML("<h3>🧠 Interactive Quality Intelligence Dashboard</h3>"),
    dept_dropdown,
    sev_dropdown,
    rc_text,
    overdue_dropdown,
    widgets.HTML("<br>"),
    predict_button
])

display(input_form, output_area)

Output()